# Baltic data — load + sanity check

Shows how to access the prepared data:
1. Connect DuckDB to the S3 AIS Parquet
2. Load criticality GeoJSON layers from `data/geo/`
3. Render a quick folium map for visual sanity

**Prerequisites:** AWS credentials configured (`aws configure`), `pip install -r requirements.txt`.

In [ ]:
import os, configparser
import duckdb
import geopandas as gpd
import folium
from pathlib import Path

cp = configparser.ConfigParser()
cp.read(os.path.expanduser('~/.aws/credentials'))
AK = cp['default']['aws_access_key_id']
SK = cp['default']['aws_secret_access_key']
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f"CREATE SECRET (TYPE s3, KEY_ID '{AK}', SECRET '{SK}', REGION 'eu-west-3')")

In [ ]:
# Read one day of Danish AIS directly from S3 (zero local download)
S3_DAY = "s3://edth2026-baltic/ais/parquet/source=danish/year=2024/month=12/day=25/part-*.parquet"
df = con.execute(f"""
    SELECT MMSI, Name, \"Ship type\", Timestamp, Latitude, Longitude, SOG, COG, Heading, IMO, ts
    FROM read_parquet('{S3_DAY}')
""").df()
print(f'rows: {len(df):,}, unique vessels: {df["MMSI"].nunique():,}')
df.head(5)

In [ ]:
# Load criticality GeoJSON layers
GEO = Path('../../geo')
cables = gpd.read_file(GEO / 'submarine_cables.geojson')
power_cables = gpd.read_file(GEO / 'submarine_power_cables.geojson')
pipelines = gpd.read_file(GEO / 'pipelines.geojson')
print(f'cables: {len(cables)}, power: {len(power_cables)}, pipelines: {len(pipelines)}')

In [ ]:
# Sanity map: cables + a sample of AIS points
m = folium.Map(location=[58.0, 19.0], zoom_start=6, tiles='CartoDB positron')
folium.GeoJson(cables.to_json(), name='Submarine cables',
               style_function=lambda f: {'color': '#888', 'weight': 1}).add_to(m)
folium.GeoJson(power_cables.to_json(), name='Power cables',
               style_function=lambda f: {'color': '#cc6600', 'weight': 2}).add_to(m)
# Sample AIS points (every Nth row to keep the map fast)
for _, r in df.iloc[::500].iterrows():
    folium.CircleMarker([r['Latitude'], r['Longitude']], radius=1,
                        color='red', fill=True, fill_opacity=0.5).add_to(m)
folium.LayerControl().add_to(m)
m